## 1. Dataset Preparation (Sliding Window)

In [4]:
!pip install yfinance
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler

# 1. Download historical stock data (e.g., Apple)
df = yf.download('AAPL', start='2024-01-01', end='2026-01-01', progress=False)
prices = df['Close'].values.reshape(-1, 1)

# 2. Normalize data between 0 and 1
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_prices = scaler.fit_transform(prices)

# 3. Create sliding windows (sequence length = 10)
def create_sequences(data, seq_length=10):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

SEQ_LENGTH = 10
X, y = create_sequences(scaled_prices, SEQ_LENGTH)

# Train/Test split shapes check: X is (samples, 10, 1)
print(f"Dataset shape - X: {X.shape}, y: {y.shape}")

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 372.8 kB/s eta 0:00:04
   ---------- ----------------------------- 0.5/2.0 MB 372.8 kB/s eta 0:00:04
   ---------- ----------------------------- 0.5/2.0 MB 372.8 kB/s eta 0:00:04
   --------------- ------------------------ 0.8/2.0 MB 399.5 kB/s eta 0:00:03
   --------------- ------------------------ 0.8/2.0 MB 399.5 kB/s eta 0:00:03
   --------------- ------------------------ 0.8/2.0 MB 399.5 kB/s eta 0:00:03
   --------------------- ------------------ 1.0/2


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
$AAPL: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-01-01)

1 Failed download:
['AAPL']: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-01-01)


ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by MinMaxScaler.

## 2. GRU Model Training

In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
import time

# 1. Generate dataset and create sliding windows
np.random.seed(42)
days = 500
random_walk = np.random.randn(days, 1) * 2 + 0.1
prices = np.cumsum(random_walk) + 150  # Start around $150
prices = prices.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_prices = scaler.fit_transform(prices)

def create_sequences(data, seq_length=10):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

SEQ_LENGTH = 10
X, y = create_sequences(scaled_prices, SEQ_LENGTH)

# 2. Build and Train GRU Model (using modern Keras Input layer)
gru_model = tf.keras.Sequential([
    tf.keras.Input(shape=(SEQ_LENGTH, 1)),
    tf.keras.layers.GRU(32),
    tf.keras.layers.Dense(1)
])
gru_model.compile(optimizer='adam', loss='mean_squared_error')

print("--- Training GRU Model ---")
start_time = time.time()
gru_history = gru_model.fit(X, y, epochs=10, batch_size=32, verbose=1)
gru_time = time.time() - start_time
gru_final_loss = gru_history.history['loss'][-1]

--- Training GRU Model ---
Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.0679
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0152
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0070  
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0033
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0016  
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 9.1352e-04 
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 8.0725e-04
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 7.9102e-04  
Epoch 9/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 7.9914e-04 
Epoch 10/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 7.7025e-04 

--- Training LSTM Model ---
Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.1345
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0167  
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0083 
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.00

## 3. LSTM Model Training and Performance Tracking

In [8]:
import time

# Build LSTM Model (same architecture structure)
lstm_model = tf.keras.Sequential([
    tf.keras.layers.LSTM(32, input_shape=(SEQ_LENGTH, 1)),
    tf.keras.layers.Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mean_squared_error')

# Train LSTM Model and measure time
print("\n--- Training LSTM Model ---")
start_time = time.time()
lstm_history = lstm_model.fit(X, y, epochs=10, batch_size=32, verbose=1)
lstm_time = time.time() - start_time

# Extract final metrics
gru_final_loss = gru_history.history['loss'][-1]
lstm_final_loss = lstm_history.history['loss'][-1]
gru_params = gru_model.count_params()
lstm_params = lstm_model.count_params()


--- Training LSTM Model ---
Epoch 1/10


C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2033
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0234 
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0113 
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0063 
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0039 
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0028 
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0023 
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0022
Epoch 9/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0021
Epoch 10/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0020 


## 4. Print Summary Comparison

In [9]:

print("\n" + "="*30 + " RESULTS SUMMARY " + "="*30)
print(f"GRU  -> Time: {gru_time:.4f}s | Final Loss: {gru_final_loss:.5f} | Parameters: {gru_model.count_params()}")
print(f"LSTM -> Time: {lstm_time:.4f}s | Final Loss: {lstm_final_loss:.5f} | Parameters: {lstm_model.count_params()}")
print("="*77)


============================== RESULTS SUMMARY ==============================
GRU  -> Time: 4.8492s | Final Loss: 0.00077 | Parameters: 3393
LSTM -> Time: 3.9515s | Final Loss: 0.00201 | Parameters: 4385
